# Transformer from scratch

In [1]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"

## Tokenizer Vocab

In [2]:
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
)

sorted_vocab = sorted(list(reference_gpt2.tokenizer.vocab.items()), key=lambda n: n[1])

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
rprint("The first are just single character / symble tokens", sorted_vocab[:10])
rprint("Then simple len == 2 combinations starts popping out, with some very common conjunctions and prepositions ex:", sorted_vocab[256:258] + [sorted_vocab[262]] + [sorted_vocab[265]])
rprint("Later the first words start popping out", sorted_vocab[990:997])

The first are just single character / symble tokens
[('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9)]

Then simple len == 2 combinations starts popping out, with some very common conjunctions and prepositions ex:
[('Ġt', 256), ('Ġa', 257), ('Ġthe', 262), ('at', 265)]

Later the first words start popping out
[('Ġprodu', 990), ('Ġstill', 991), ('led', 992), ('ah', 993), ('Ġhere', 994), ('Ġworld', 995), ('Ġthough', 996)]

In [4]:
rprint(f"The vocab size {len(sorted_vocab)} is given by 256 (UTF-8 raw bytes) + 50.000 BPE (Byte Pair Encoding) merge operations + {sorted_vocab[50256]}.")


The vocab size 50257 is given by 256 (UTF-8 raw bytes) + 50.000 BPE (Byte Pair Encoding) merge operations + 
('<|endoftext|>', 50256).

In [5]:
rprint("Same word different encodings")
words = ["Ralph", " Ralph", " ralph", "ralph"]

for w in words:
  print(f"'{w}' -> {reference_gpt2.to_str_tokens(w)}")

Same word different encodings

'Ralph' -> ['<|endoftext|>', 'R', 'alph']
' Ralph' -> ['<|endoftext|>', ' Ralph']
' ralph' -> ['<|endoftext|>', ' r', 'alph']
'ralph' -> ['<|endoftext|>', 'ral', 'ph']


## Implementation

### Config

In [6]:
rprint(reference_gpt2.cfg)

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='cpu'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 84934656,
 'normalization_type': 'LN',
 'num_experts': None,
 'original_architecture': 'GPT2LMHeadModel',
 'output_logits_soft_cap': -1.0,
 'parallel_attn_mlp': False,
 'positional_embedding_type': 'standard',
 'post_embedding_ln': False,
 'relative_attention_max_distance': None,
 'relative_attention_num_buckets': None,
 'rotary_adjacent_pairs': False,
 'rotary_base': 10000,
 'rotary_base_local': None,
 'rotary_dim': None,
 'scale_attn_by_inverse_layer_idx': False,
 'seed': None,
 'tie_word_embeddings': False,
 'tokenizer_name': 'gpt2',
 'tokenizer_prepends_bos': False,
 'trust_remote_code': False,
 'ungroup_grouped_query_attention': False,
 'use_NTK_by_parts_rope': False,
 'use_attn_in': False,
 'use_attn_result': False,
 'use_attn_scale': True,
 'use_hook_mlp_in': False,
 'use_hook_tokens': False,
 'use_local_attn': False,
 'use_normalization_before_and_after': False,
 'use_qk_norm': False,
 'use_split_qkv_input': False,
 'window_size': None}

In [7]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12


cfg = Config()
rprint(cfg)

Config(
    d_model=768,
    debug=True,
    layer_norm_eps=1e-05,
    d_vocab=50257,
    init_range=0.02,
    n_ctx=1024,
    d_head=64,
    d_mlp=3072,
    n_heads=12,
    n_layers=12
)

### tests utils

In [93]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).to(device)
logits, cache = reference_gpt2.run_with_cache(tokens)

In [90]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randn(shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")


def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randint(100, 1000, shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")


def load_gpt2_test(cls, gpt2_layer, input):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    print("Input shape:", input.shape)
    orig_input = input.clone()
    output = layer(orig_input)
    assert t.allclose(input, orig_input), "Input has been modified, make sure operations are not done in place"
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape)
    try:
        reference_output = gpt2_layer(input)
    except:
        reference_output = gpt2_layer(input, input, input)
    print("Reference output shape:", reference_output.shape, "\n")
    comparison = t.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum() / comparison.numel():.2%} of the values are correct\n")
    assert 1 - (comparison.sum() / comparison.numel()) < 1e-5, "More than 0.01% of the values are incorrect"

### Single Bricks

#### Layer Norm
[pytorch reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

In [97]:
class LayerNorm(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.w = nn.Parameter(t.ones(cfg.d_model))
    self.b = nn.Parameter(t.zeros(cfg.d_model))

  def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
    mean = t.mean(residual, dim=-1, keepdim =True)
    variance = t.var(residual, dim=-1, keepdim=True, unbiased=False)
    normalized = (residual - mean) / t.sqrt(variance + self.cfg.layer_norm_eps)

    if self.cfg.debug:
      print("Mean:")
      rprint(mean.shape)
      print("Variance:")
      rprint(variance.shape)
      print("Normalized:")
      rprint(normalized.shape)

    return normalized * self.w + self.b


rand_float_test(LayerNorm, [2, 4, 768])
load_gpt2_test(LayerNorm, reference_gpt2.ln_final, cache["resid_post", 11])

Input shape: torch.Size([2, 4, 768])
Mean:


torch.Size([2, 4, 1])

Variance:


torch.Size([2, 4, 1])

Normalized:


...

Variance:


torch.Size([1, 35, 1])

Normalized:


torch.Size([1, 35, 768])

Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### Embed

look up table, given the token index return the token representation

In [98]:
class Embed(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
    nn.init.normal_(self.W_E, std=self.cfg.init_range)
  
  def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:    
    if self.cfg.debug:
      rprint(tokens)
      rprint(self.W_E.shape)

    return self.W_E[tokens]

rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

Input shape: torch.Size([2, 4])


tensor([[456, 776, 967, 763],
        [512, 124, 664, 156]])

torch.Size([50257, 768])

Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35])


tensor([[50256,    40,   716,   281,  4998,  1960,   382, 19741,    11,   875,
         12342,    12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,
            13,  1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,
          1011,   625,   262,   995,     0]])

torch.Size([50257, 768])

Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### Position Embed
add positional information to the embeddings
attention is position independent 

In [99]:
class PosEmbed(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
    nn.init.normal_(self.W_pos, std=self.cfg.init_range)

  def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
    batch, seq_len = tokens.shape

    return einops.repeat(self.W_pos[:seq_len], "seq d_model -> batch seq d_model", batch=batch)

rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### Attention

In [15]:
def rand_causal_mask_test(cls, shape):
  cfg = Config(debug=True)
  layer = cls(cfg).to(device)
  random_input = t.randn(shape).to(device)
  print("Input shape:", random_input.shape)
  output = layer.apply_causal_mask(random_input)
  if isinstance(output, tuple):
      output = output[0]
  print("Output shape:", output.shape, "\n")


In [100]:
class Attention(nn.Module):
  IGNORE: Float[Tensor, ""]

  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg

    self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
    self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
    self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
    self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
    self.b_O = nn.Parameter(t.zeros((cfg.d_model)))

    nn.init.normal_(self.W_Q, std=self.cfg.init_range)
    nn.init.normal_(self.W_K, std=self.cfg.init_range)
    nn.init.normal_(self.W_V, std=self.cfg.init_range)
    nn.init.normal_(self.W_O, std=self.cfg.init_range)

    self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

  def apply_causal_mask(
    self,
    attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"],
  ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
    """
    Applies a causal mask to attention scores, and returns masked scores.
    """
    mask = t.ones(attn_scores.shape[-2:], dtype=t.bool, device=attn_scores.device).triu(diagonal=1)
    
    if self.cfg.debug:
      rprint(mask)

    return attn_scores.masked_fill(mask, self.IGNORE)

  def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:

    Q = einops.einsum(normalized_resid_pre, self.W_Q, "batch posn d_model, n_heads d_model d_head -> batch n_heads posn d_head") + self.b_Q[None, :, None, :]
    K = einops.einsum(normalized_resid_pre, self.W_K, "batch posn d_model, n_heads d_model d_head -> batch n_heads posn d_head") + self.b_K[None, :, None, :]
    V = einops.einsum(normalized_resid_pre, self.W_V, "batch posn d_model, n_heads d_model d_head -> batch n_heads posn d_head") + self.b_V[None, :, None, :]
    attention_scores = self.apply_causal_mask(Q @ K.mT)
    attention = t.softmax(attention_scores / self.cfg.d_head ** 0.5, axis=-1) @ V

    z_proj = einops.einsum(attention, self.W_O, "batch n_heads posn d_head, n_heads d_head d_model -> batch posn d_model") + self.b_O[None, None, :]

    if self.cfg.debug:
      rprint(f"query: {Q.shape}")
      rprint(f"key: {K.shape}")
      rprint(f"value: {V.shape}")
      rprint(f"attention_scores: {attention_scores.shape}")
      rprint(f"attention: {attention.shape}")
      rprint(f"z_proj: {z_proj.shape}")

    return z_proj


att = Attention(cfg)
rand_causal_mask_test(Attention, (2, cfg.n_heads, cfg.d_head, cfg.d_head))
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["normalized", 0, "ln1"])

Input shape: torch.Size([2, 12, 64, 64])


tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

Output shape: torch.Size([2, 12, 64, 64]) 

Input shape: torch.Size([2, 4, 768])


tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

query: torch.Size([2, 12, 4, 64])

...

value: torch.Size([1, 12, 35, 64])

attention_scores: torch.Size([1, 12, 35, 35])

attention: torch.Size([1, 12, 35, 64])

z_proj: torch.Size([1, 35, 768])

Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### MLP

In [101]:
class MLP(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
    self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
    self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
    self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
    nn.init.normal_(self.W_in, std=self.cfg.init_range)
    nn.init.normal_(self.W_out, std=self.cfg.init_range)

  def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
    w_sum = einops.einsum(normalized_resid_mid, self.W_in, "batch posn d_model, d_model d_mlp -> batch posn d_mlp") + self.b_in
    hidden_state = gelu_new(w_sum)
    out = einops.einsum(hidden_state, self.W_out, "batch posn d_mlp, d_mlp d_model -> batch posn d_model") + self.b_out

    if self.cfg.debug:
      rprint(f"w_sum: {w_sum.shape}")
      rprint(f"hidden_state: {hidden_state.shape}")
      rprint(f"out: {out.shape}")

    return out

rand_float_test(MLP, [2, 4, 768])
load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["normalized", 0, "ln2"])

Input shape: torch.Size([2, 4, 768])


w_sum: torch.Size([2, 4, 3072])

hidden_state: torch.Size([2, 4, 3072])

out: torch.Size([2, 4, 768])

Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35, 768])


w_sum: torch.Size([1, 35, 3072])

hidden_state: torch.Size([1, 35, 3072])

out: torch.Size([1, 35, 768])

Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### TransformerBlock

In [103]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.ln1 = LayerNorm(cfg)
    self.attn = Attention(cfg)
    self.ln2 = LayerNorm(cfg)
    self.mlp = MLP(cfg)

  def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
    multi_attention = self.attn(self.ln1(resid_pre)) + resid_pre
    resid_post = self.mlp(self.ln2(multi_attention)) + multi_attention

    if self.cfg.debug:
      rprint(f"multi_attention: {multi_attention.shape}")
      rprint(f"resid_post: {resid_post.shape}")
    
    return resid_post


rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

Input shape: torch.Size([2, 4, 768])
Mean:


torch.Size([2, 4, 1])

Variance:


torch.Size([2, 4, 1])

Normalized:


...

hidden_state: torch.Size([1, 35, 3072])

out: torch.Size([1, 35, 768])

multi_attention: torch.Size([1, 35, 768])

resid_post: torch.Size([1, 35, 768])

Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



#### Unembed

In [104]:
class Unembed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
    nn.init.normal_(self.W_U, std=self.cfg.init_range)
    self.b_U = nn.Parameter(t.zeros((cfg.d_vocab), requires_grad=False))

  def forward(self, normalized_resid_final: Float[Tensor, "batch position d_model"]
             ) -> Float[Tensor, "batch position d_vocab"]:
    out = einops.einsum(normalized_resid_final, self.W_U, "batch position d_model, d_model d_vocab -> batch position d_vocab") + self.b_U

    if self.cfg.debug:
      rprint(f"out: {out.shape}")

    return out


rand_float_test(Unembed, [2, 4, 768])
load_gpt2_test(Unembed, reference_gpt2.unembed, cache["ln_final.hook_normalized"])

Input shape: torch.Size([2, 4, 768])


out: torch.Size([2, 4, 50257])

Output shape: torch.Size([2, 4, 50257]) 

Input shape: torch.Size([1, 35, 768])


out: torch.Size([1, 35, 50257])

Output shape: torch.Size([1, 35, 50257])
Reference output shape: torch.Size([1, 35, 50257]) 

100.00% of the values are correct



### Complete Implementation

In [106]:
class DemoTransformer(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.cfg = cfg
    self.embed = Embed(cfg)
    self.pos_embed = PosEmbed(cfg)
    self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg.n_layers)])
    self.ln_final = LayerNorm(cfg)
    self.unembed = Unembed(cfg)

  def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
    
    embedding = self.embed(tokens) + self.pos_embed(tokens)
    resid = self.blocks(embedding)
    normalized = self.ln_final(resid)
    logits = self.unembed(normalized)

    if self.cfg.debug:
      rprint(f"embedding: {embedding.shape}")
      rprint(f"resid: {resid.shape}")
      rprint(f"normalized: {normalized.shape}")
      rprint(f"logits: {logits.shape}")

    return logits

rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

Input shape: torch.Size([2, 4])


tensor([[270, 765, 474, 585],
        [694, 242, 677, 690]])

torch.Size([50257, 768])

Mean:


torch.Size([2, 4, 1])

...

embedding: torch.Size([1, 35, 768])

resid: torch.Size([1, 35, 768])

normalized: torch.Size([1, 35, 768])

logits: torch.Size([1, 35, 50257])

Output shape: torch.Size([1, 35, 50257])
Reference output shape: torch.Size([1, 35, 50257]) 

100.00% of the values are correct



In [108]:
demo_gpt2 = DemoTransformer(Config(debug=False)).to(device)
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)

demo_logits = demo_gpt2(tokens)
rprint(demo_logits)

tensor([[[ -43.4317,  -39.8364,  -43.0659,  ...,  -54.0877,  -54.3452,
           -42.3645],
         [-128.0392, -127.9935, -130.7010,  ..., -136.7122, -129.9262,
          -129.3966],
         [-119.8521, -121.0064, -123.8820,  ..., -128.5180, -126.6027,
          -121.9061],
         ...,
         [-112.9815, -112.7749, -117.0633,  ..., -121.2914, -117.6574,
          -114.5005],
         [ -98.6724, -104.4888, -108.7361,  ..., -118.3552, -113.8766,
          -106.3603],
         [-126.8285, -128.9596, -128.3941,  ..., -140.1971, -138.5883,
          -122.3698]]], grad_fn=<AddBackward0>)

In [109]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # Get logprobs the first seq_len-1 predictions (so we can compare them with the actual next tokens)
    log_probs_for_tokens = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

    return log_probs_for_tokens


pred_log_probs = get_log_probs(demo_logits, tokens)
print(f"Avg cross entropy loss: {-pred_log_probs.mean():.4f}")
print(f"Avg cross entropy loss for uniform distribution: {math.log(demo_gpt2.cfg.d_vocab):4f}")
print(f"Avg probability assigned to correct token: {pred_log_probs.exp().mean():4f}")


Avg cross entropy loss: 4.5647
Avg cross entropy loss for uniform distribution: 10.824905
Avg probability assigned to correct token: 0.087910


In [110]:
test_string = """Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as"""
for i in tqdm(range(100)):
    test_tokens = reference_gpt2.to_tokens(test_string).to(device)
    demo_logits = demo_gpt2(test_tokens)
    test_string += reference_gpt2.tokenizer.decode(demo_logits[-1, -1].argmax())

print(test_string)


  0%|          | 0/100 [00:00<?, ?it/s]

Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as climate change, the spread of infectious diseases and the spread of infectious diseases.


The research is published in the journal Nature Communications.


The research team is led by Dr. Michael J. H. Haldane, a professor of biology at the University of California, Berkeley, and co-author of the paper.


"We are very excited to see that the AI community is starting to take notice of the potential for AI to be a major threat to the human race,"


## Training

## Sampling

In [ ]:
class TransformerSampler:
  def __init__(self, model: DemoTransformer, tokenizer: GPT2TokenizerFast):
    self.model = model
    self.cfg = model.cfg
    self.tokenizer = tokenizer

  @t.inference_mode()
  def sample(self, prompt: str, max_tokens_generated=100, verbose=False, **kwargs) -> str:
    """
    Returns a string of autoregressively generated text, starting from the prompt
    Sampling terminates at max_tokens_generated, or when the model generates an end-of-sequence token. kwargs are
    passed to sample_next_token, to give detailed instructions on how new tokens are chosen.
    """

    input_tokens = self.tokenizer.encode(prompt)
    print(input_tokens)
        
    return prompt

    @staticmethod
    def sample_next_token(
        input_ids: Int[Tensor, " seq_len"],
        logits: Float[Tensor, "d_vocab"],
        temperature=1.0,
        top_k=0,
        top_p=0.0,
        frequency_penalty=0.0,
        seed=None,
    ) -> int:
        assert input_ids.ndim == 1, "input_ids should be a 1D sequence of token ids"
        assert temperature >= 0, "Temperature should be non-negative"
        assert 0 <= top_p <= 1.0, "Top-p must be a probability"
        assert 0 <= top_k, "Top-k must be non-negative"
        assert not (top_p != 0 and top_k != 0), "At most one of top-p and top-k supported"

        # Set random seeds for reproducibility
        if seed is not None:
            t.manual_seed(seed)
            np.random.seed(seed)

        # Apply all the specialized sampling methods
        if temperature == 0:
            return TransformerSampler.greedy_search(logits)
        elif temperature != 1.0:
            logits = TransformerSampler.apply_temperature(logits, temperature)
        if frequency_penalty != 0.0:
            logits = TransformerSampler.apply_frequency_penalty(input_ids, logits, frequency_penalty)
        if top_k > 0:
            return TransformerSampler.sample_top_k(logits, top_k)
        if top_p > 0.0:
            return TransformerSampler.sample_top_p(logits, top_p)
        return TransformerSampler.sample_basic(logits)

    @staticmethod
    def greedy_search(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Returns the most likely token (as an int).
        """
        raise NotImplementedError()

    @staticmethod
    def apply_temperature(logits: Float[Tensor, "d_vocab"], temperature: float) -> Float[Tensor, "d_vocab"]:
        """
        Applies temperature scaling to the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def apply_frequency_penalty(
        input_ids: Int[Tensor, " seq_len"], logits: Float[Tensor, "d_vocab"], freq_penalty: float
    ) -> Float[Tensor, "d_vocab"]:
        """
        Applies a frequency penalty to the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_basic(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Samples from the distribution defined by the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_top_k(logits: Float[Tensor, "d_vocab"], k: int) -> int:
        """
        Samples from the top k most likely tokens.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_top_p(logits: Float[Tensor, "d_vocab"], top_p: float, min_tokens_to_keep: int = 1) -> int:
        """
        Samples from the most likely tokens which make up at least p cumulative probability.
        """
        raise NotImplementedError()

    @t.inference_mode()
    def beam_search(
        self,
        prompt: str,
        num_return_sequences: int,
        num_beams: int,
        max_new_tokens: int,
        no_repeat_ngram_size: int | None = None,
    ) -> list[tuple[float, str]]:
        """
        Implements a beam search, by repeatedly performing the `generate` and `filter` steps (starting from the initial
        prompt) until either of the two stopping criteria are met: (1) we've generated `max_new_tokens` tokens, or (2)
        we've generated `num_returns_sequences` terminating sequences.
        """
        raise NotImplementedError()


t.set_grad_enabled(False)  # gradients are not necessary for sampling

model = DemoTransformer(Config(debug=False)).to(device)
model.load_state_dict(reference_gpt2.state_dict(), strict=False)
tokenizer = reference_gpt2.tokenizer
sampler = TransformerSampler(model, tokenizer)

prompt = "Jingle bells, jingle bells, jingle all the way"
print(f"Testing greedy decoding\nPrompt:   {prompt!r}")

expected = "Jingle bells, jingle bells, jingle all the way up to the top of the mountain."
output = sampler.sample(prompt, max_tokens_generated=8, temperature=0.0)

# print(f"Expected: {expected!r}\nActual:   {output!r}\n")
# assert output == expected

# print("Tests passed!")
